# Event 4245 live-replay validation

Compare the live-captured detector rows with the known event-4245 reference CSV. The source-specific `packet_file` column is intentionally excluded.

In [ ]:
from pathlib import Path
import pandas as pd

reference_path = Path(
    "/home/fabric/work/visualization-wibwiz/graph_notebook/4245.csv"
)
live_path = Path("/tmp/replay-live-current.csv.gz")
columns = [
    "seq_id",
    "timestamp",
    "tick",
    "adc_value",
    "wire_id",
    "tpc",
    "slot",
    "plane",
]

for path in [reference_path, live_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

reference = pd.read_csv(reference_path, usecols=columns).astype("int64")
live = pd.read_csv(
    live_path, compression="gzip", usecols=columns
).astype("int64")
print("Reference rows:", len(reference))
print("Live rows:     ", len(live))

reference_counts = reference.value_counts(sort=False).rename("reference")
live_counts = live.value_counts(sort=False).rename("live")
comparison = (
    pd.concat([reference_counts, live_counts], axis=1)
    .fillna(0)
    .astype(int)
)
mismatches = comparison[comparison["reference"] != comparison["live"]]

if mismatches.empty:
    print("PASS: Every detector row matches the reference CSV.")
else:
    print("FAIL:", len(mismatches), "row patterns differ.")
    display(mismatches.head(20))
